# HERCULEAN Data Cleaning
This notebook loads, inspects, cleans, and verifies the Herculean data for our project.

## 1. State the project vision and question
### Project Vision
Our project aims to help an investor-relations analyst identify unusually wide daily price movements across eight large-cap companies and review relevant news and filing information around those movements for human follow-up.

### Business Question
Which companies and dates have unusually wide daily price ranges, and what news highlights and recent 10-K or 10-Q filing information are available around those movements?

### Main Metric
The main metric is daily price range, calculated as daily high price − daily low price and measured in U.S. dollars per share.

## 2. Define one row

**Prices:** One row represents one company's daily stock price and trading volume on one trading date.

**News:** One row represents the news highlights for one company on one date, not necessarily one individual news article.

**Filings:** One row represents one 10-K or 10-Q filing for one company on one filing date, including extracted MD&A and risk-factor content.



## 3. Preserve the source

The original data consists of three Parquet files: `filings.parquet`, `news.parquet`, and `prices.parquet`. The data was downloaded from the HERCULEAN dataset by TheFinAI on Hugging Face.
The original datasets are loaded without modifying the source files. Separate copies (`filing_new`, `news_new`, and `price_new`) are created for cleaning so that the original datasets remain unchanged.

**Source:** TheFinAI/Herculean on Hugging Face  

**Download date:** [October 2, 2026]

In [1]:
import pandas as pd

filings = pd.read_parquet('Data/filings.parquet')
news = pd.read_parquet('Data/news.parquet')
prices = pd.read_parquet('Data/prices.parquet')

display(filings.head())
display(news.head())
display(prices.head())

,id,symbol,date,document_type,mda_content,risk_content
0,1,AAPL,2024-02-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
1,2,AAPL,2024-05-02,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
2,3,AAPL,2024-08-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
3,4,AAPL,2024-11-01,10-K,Item 7. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
4,5,AAPL,2025-01-31,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...


,id,symbol,date,highlights
0,1,AAPL,2024-12-01,"Apple’s December 1, 2024 coverage centers on i..."
1,2,AAPL,2024-12-02,"The December 2, 2024 Apple coverage portrays a..."
2,3,AAPL,2024-12-03,"demonstrated strong financial performance, wit..."
3,4,AAPL,2024-12-04,Across the December 3 coverage compiled for Ap...
4,5,AAPL,2024-12-05,- The Shacknews report highlights Apple’s firs...


,id,symbol,date,open,high,low,close,adj_close,volume
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000
2,3,AAPL,2024-12-04,242.869995,244.110001,241.250000,243.009995,241.689728,44383900
3,4,AAPL,2024-12-05,243.990005,244.539993,242.130005,243.039993,241.719559,40033900
4,5,AAPL,2024-12-06,242.910004,244.630005,242.080002,242.839996,241.520630,36870600


In [2]:
filing_new = filings.copy()
news_new = news.copy()
price_new = prices.copy()

display(filing_new.head())
display(news_new.head())
display(price_new.head())

,id,symbol,date,document_type,mda_content,risk_content
0,1,AAPL,2024-02-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
1,2,AAPL,2024-05-02,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
2,3,AAPL,2024-08-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
3,4,AAPL,2024-11-01,10-K,Item 7. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
4,5,AAPL,2025-01-31,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...


,id,symbol,date,highlights
0,1,AAPL,2024-12-01,"Apple’s December 1, 2024 coverage centers on i..."
1,2,AAPL,2024-12-02,"The December 2, 2024 Apple coverage portrays a..."
2,3,AAPL,2024-12-03,"demonstrated strong financial performance, wit..."
3,4,AAPL,2024-12-04,Across the December 3 coverage compiled for Ap...
4,5,AAPL,2024-12-05,- The Shacknews report highlights Apple’s firs...


,id,symbol,date,open,high,low,close,adj_close,volume
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000
2,3,AAPL,2024-12-04,242.869995,244.110001,241.250000,243.009995,241.689728,44383900
3,4,AAPL,2024-12-05,243.990005,244.539993,242.130005,243.039993,241.719559,40033900
4,5,AAPL,2024-12-06,242.910004,244.630005,242.080002,242.839996,241.520630,36870600


In [3]:
#Verify original date types unchanged
print("ORIGINAL DATE TYPES")
print("Filings:", filings["date"].dtype)
print("News:", news["date"].dtype)
print("Prices:", prices["date"].dtype)

ORIGINAL DATE TYPES
Filings: object
News: object
Prices: object


## 4. Inspect columns and types

The original datasets were inspected for their number of rows, columns, and data types. The date column in all three datasets is stored as an object and should be converted to datetime for date-based analysis. The remaining column data types are appropriate and are retained without conversion.

The existing rules below were written from the saved inspection results. Only date conversion changes the working data. `errors="coerce"` would produce `NaT` for an unparseable value; any newly observed failures must be investigated and documented rather than silently discarded.


In [4]:
print('Filings shape:', filings.shape)
print('News shape:', news.shape)
print('Prices shape:', prices.shape)

print("\nFILINGS")
filings.info()

print("\nNEWS")
news.info()

print("\nPRICES")
prices.info()

Filings shape: (73, 6)
News shape: (3888, 4)
Prices shape: (2656, 9)

FILINGS
<class 'pandas.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             73 non-null     int32 
 1   symbol         73 non-null     str   
 2   date           73 non-null     object
 3   document_type  73 non-null     str   
 4   mda_content    73 non-null     str   
 5   risk_content   73 non-null     str   
dtypes: int32(1), object(1), str(4)
memory usage: 7.7+ MB

NEWS
<class 'pandas.DataFrame'>
RangeIndex: 3888 entries, 0 to 3887
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          3888 non-null   int32 
 1   symbol      3888 non-null   str   
 2   date        3888 non-null   object
 3   highlights  3888 non-null   str   
dtypes: int32(1), object(1), str(2)
memory usage: 17.8+ MB

PRICES
<class 'pandas.DataFr

### Existing cleaning rules

#### Based on Data Inspection, the following cleaning rules will be applied:
1. Convert the `date` column in all three datasets to datetime format because it is currently stored as an object.
2. Keep all rows because no missing values, exact duplicate rows, repeated IDs, or repeated symbol-date combinations were found.
3. Keep all price and volume values because no invalid or inconsistent values were found.
4. Keep the original text values because no leading or trailing spaces, blank text fields, or inconsistent symbols or document types were found.
5. Keep the original datasets unchanged and create separate copies for cleaning and analysis.

In [5]:
filing_new['date'] = pd.to_datetime(filing_new['date'], errors='coerce')
news_new['date'] = pd.to_datetime(news_new['date'], errors='coerce')
price_new['date'] = pd.to_datetime(price_new['date'], errors='coerce')

In [6]:
# Check date data types after cleaning
print('DATE DATA TYPES AFTER CLEANING')
print("Filings:", filing_new["date"].dtype)
print("News:", news_new["date"].dtype)
print("Prices:", price_new["date"].dtype)

DATE DATA TYPES AFTER CLEANING
Filings: datetime64[s]
News: datetime64[s]
Prices: datetime64[s]


In [7]:
# Verify conversion failures on the cleaned copies; do not change values.
for name, original, cleaned in [("Filings", filings, filing_new), ("News", news, news_new), ("Prices", prices, price_new)]:
    failures = (original["date"].notna() & cleaned["date"].isna()).sum()
    print(name, "failed conversions:", failures, "cleaned missing dates:", cleaned["date"].isna().sum())
    assert failures == 0, f"Investigate date conversion failures in {name}"


Filings failed conversions: 0 cleaned missing dates: 0
News failed conversions: 0 cleaned missing dates: 0
Prices failed conversions: 0 cleaned missing dates: 0


## 5. Standardize text


In [8]:
# Check leading or trailing spaces
print("LEADING/TRAILING SPACES")
print("Filings symbols:", (filings["symbol"] != filings["symbol"].str.strip()).sum())
print("News symbols:", (news["symbol"] != news["symbol"].str.strip()).sum())
print("Prices symbols:", (prices["symbol"] != prices["symbol"].str.strip()).sum())
print("Document types:", (filings["document_type"] != filings["document_type"].str.strip()).sum())

# Check blank text fields
print("\nBLANK TEXT FIELDS")
print("News highlights:", news["highlights"].str.strip().eq("").sum())
print("MDA content:", filings["mda_content"].str.strip().eq("").sum())
print("Risk content:", filings["risk_content"].str.strip().eq("").sum())

LEADING/TRAILING SPACES
Filings symbols: 0
News symbols: 0
Prices symbols: 0
Document types: 0

BLANK TEXT FIELDS
News highlights: 0
MDA content: 0
Risk content: 0


In [9]:
# Check categorical text consistency
print("UNIQUE SYMBOLS")
print("Filings:", sorted(filings["symbol"].unique()))
print("News:", sorted(news["symbol"].unique()))
print("Prices:", sorted(prices["symbol"].unique()))
print("\nUNIQUE DOCUMENT TYPES")
print(sorted(filings["document_type"].unique()))

UNIQUE SYMBOLS
Filings: ['AAPL', 'ADBE', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']
News: ['AAPL', 'ADBE', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']
Prices: ['AAPL', 'ADBE', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']

UNIQUE DOCUMENT TYPES
['10-K', '10-Q']


In [10]:
# Verify text and IDs are preserved; show illustrative unchanged text examples.
for name, original, cleaned, text_columns in [
    ("Filings", filings, filing_new, ["symbol", "document_type", "mda_content", "risk_content"]),
    ("News", news, news_new, ["symbol", "highlights"]),
    ("Prices", prices, price_new, ["symbol"]),
]:
    unchanged = original[text_columns + ["id"]].equals(cleaned[text_columns + ["id"]])
    print(name, "text and IDs unchanged:", unchanged)
    assert unchanged
    for column in text_columns:
        display(pd.DataFrame({"original": original[column].head(1), "cleaned": cleaned[column].head(1)}))


Filings text and IDs unchanged: True


,original,cleaned
0,AAPL,AAPL


,original,cleaned
0,10-Q,10-Q


,original,cleaned
0,Item 2. Management&#8217;s Discussion and Ana...,Item 2. Management&#8217;s Discussion and Ana...


,original,cleaned
0,Item 1A. Risk Factors \n\nThe Company&#8217;s...,Item 1A. Risk Factors \n\nThe Company&#8217;s...


News text and IDs unchanged: True


,original,cleaned
0,AAPL,AAPL


,original,cleaned
0,"Apple’s December 1, 2024 coverage centers on i...","Apple’s December 1, 2024 coverage centers on i..."


Prices text and IDs unchanged: True


,original,cleaned
0,AAPL,AAPL


## 6. Investigate duplicates


In [11]:
# exact duplicates rows
print("EXACT DUPLICATE ROWS")
print('Filings Duplicates:', filings.duplicated().sum())
print('News Duplicates:', news.duplicated().sum())
print('Prices Duplicates:', prices.duplicated().sum())

# Repeated IDs
print("\nREPEATED IDs")
print("Filings:", filings["id"].duplicated().sum())
print("News:", news["id"].duplicated().sum())
print("Prices:", prices["id"].duplicated().sum())

# Repeated symbol-date combinations
print("\nREPEATED SYMBOL-DATE Combinations")
print("Filings:", filings.duplicated(subset=["symbol", "date"]).sum())
print("News:", news.duplicated(subset=["symbol", "date"]).sum())
print("Prices:", prices.duplicated(subset=["symbol", "date"]).sum()) 

EXACT DUPLICATE ROWS
Filings Duplicates: 0
News Duplicates: 0
Prices Duplicates: 0

REPEATED IDs
Filings: 0
News: 0
Prices: 0

REPEATED SYMBOL-DATE Combinations
Filings: 0
News: 0
Prices: 0


## 7. Handle missing data



In [12]:
print("\nFILINGS MISSING VALUES")
print(filings.isna().sum())

print("\nNEWS MISSING VALUES")
print(news.isna().sum())

print("\nPRICES MISSING VALUES")
print(prices.isna().sum())


FILINGS MISSING VALUES
id               0
symbol           0
date             0
document_type    0
mda_content      0
risk_content     0
dtype: int64

NEWS MISSING VALUES
id            0
symbol        0
date          0
highlights    0
dtype: int64

PRICES MISSING VALUES
id           0
symbol       0
date         0
open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64


In [13]:
for name, original, cleaned in [("Filings", filings, filing_new), ("News", news, news_new), ("Prices", prices, price_new)]:
    print(name)
    display(pd.DataFrame({"before": original.isna().sum(), "after": cleaned.isna().sum()}))


Filings


,before,after
id,0,0
symbol,0,0
date,0,0
document_type,0,0
mda_content,0,0
risk_content,0,0


News


,before,after
id,0,0
symbol,0,0
date,0,0
highlights,0,0


Prices


,before,after
id,0,0
symbol,0,0
date,0,0
open,0,0
high,0,0
low,0,0
close,0,0
adj_close,0,0
volume,0,0


## 8. Check suspicious values


In [14]:
# Check for invalid dates
print("INVALID DATES")
print("Filings:", pd.to_datetime(filings["date"], errors="coerce").isna().sum())
print("News:", pd.to_datetime(news["date"], errors="coerce").isna().sum())
print("Prices:", pd.to_datetime(prices["date"], errors="coerce").isna().sum())

# Check impossible or inconsistent price values
print("\nNON-POSITIVE PRICES")
print((prices[["open", "high", "low", "close", "adj_close"]] <= 0).sum())
print("\nNEGATIVE VOLUME")
print((prices["volume"] < 0).sum())
print("\nHIGH LOWER THAN LOW")
print((prices["high"] < prices["low"]).sum())
print("\nOPEN OUTSIDE DAILY RANGE")
print(((prices["open"] < prices["low"]) |
       (prices["open"] > prices["high"])).sum())
print("\nCLOSE OUTSIDE DAILY RANGE")
print(((prices["close"] < prices["low"]) |
       (prices["close"] > prices["high"])).sum())

INVALID DATES
Filings: 0
News: 0
Prices: 0

NON-POSITIVE PRICES
open         0
high         0
low          0
close        0
adj_close    0
dtype: int64

NEGATIVE VOLUME
0

HIGH LOWER THAN LOW
0

OPEN OUTSIDE DAILY RANGE
0

CLOSE OUTSIDE DAILY RANGE
0


## 9. Verify merges

Prices and news are merged on symbol and date using a one-to-one left join. There are 2,656 price records before the merge and 2,656 records after the merge, so the merge does not create additional rows. All price records have matching news records, with 0 unmatched price records.

In [15]:
price_news = price_new.merge(
    news_new,
    on=["symbol", "date"],
    how="left",
    validate="one_to_one",
    suffixes=("_price", "_news")
)

price_news

,id_price,symbol,date,open,high,low,close,adj_close,volume,id_news,highlights
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100,2,"The December 2, 2024 Apple coverage portrays a..."
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000,3,"demonstrated strong financial performance, wit..."
2,3,AAPL,2024-12-04,242.869995,244.110001,241.250000,243.009995,241.689728,44383900,4,Across the December 3 coverage compiled for Ap...
3,4,AAPL,2024-12-05,243.990005,244.539993,242.130005,243.039993,241.719559,40033900,5,- The Shacknews report highlights Apple’s firs...
4,5,AAPL,2024-12-06,242.910004,244.630005,242.080002,242.839996,241.520630,36870600,6,"Apple Inc. is portrayed in the December 6, 202..."
...,...,...,...,...,...,...,...,...,...,...,...
2651,2652,TSLA,2026-03-25,389.989990,396.230011,385.010010,385.950012,385.950012,55157300,3882,A cluster of TSLA-relevant coverage on 2026-03...
2652,2653,TSLA,2026-03-26,381.600006,384.440002,371.869995,372.109985,372.109985,55522900,3883,Across the set of TSLA-focused coverage on 202...
2653,2654,TSLA,2026-03-27,369.690002,369.859985,359.470001,361.829987,361.829987,62065700,3884,Tesla-related coverage on 2026-03-27 presents ...
2654,2655,TSLA,2026-03-30,365.859985,367.290009,352.140015,355.279999,355.279999,67954400,3887,"The collection of March 30, 2026 articles pres..."


In [16]:
print("Price rows before merge:", len(price_new))
print("Rows after merge:", len(price_news))
print("Price rows without matching news:", price_news["highlights"].isna().sum())

Price rows before merge: 2656
Rows after merge: 2656
Price rows without matching news: 0


## 10. Reconcile changes

The cleaning process did not add or remove any records. Filings remained at 73 rows, news at 3,888 rows, and prices at 2,656 rows. All non-date values remained unchanged, and total trading volume remained 159,456,800,900 before and after cleaning. Therefore, the only change to the source data was converting the date columns to datetime format.

In [17]:
row_counts = pd.DataFrame([
    {"table": name, "starting_rows": len(original), "final_rows": len(cleaned), "difference": len(cleaned) - len(original)}
    for name, original, cleaned in [("Filings", filings, filing_new), ("News", news, news_new), ("Prices", prices, price_new)]
])
display(row_counts)
for name, original, cleaned in [("Filings", filings, filing_new), ("News", news, news_new), ("Prices", prices, price_new)]:
    unchanged_columns = original.columns.drop("date")
    unchanged = original[unchanged_columns].equals(cleaned[unchanged_columns])
    print(name, "non-date values unchanged:", unchanged)
    assert len(original) == len(cleaned) and unchanged
print("Volume totals before/after:", prices["volume"].sum(), price_new["volume"].sum())


,table,starting_rows,final_rows,difference
0,Filings,73,73,0
1,News,3888,3888,0
2,Prices,2656,2656,0


Filings non-date values unchanged: True
News non-date values unchanged: True
Prices non-date values unchanged: True
Volume totals before/after: 159456800900 159456800900


## 11. Check calculations

Daily range is calculated as high - low. All 2,656 price records have valid high and low values, so all 2,656 records are included in the calculation. There are no missing daily ranges, symbols, or dates. Therefore, missing values do not affect the calculation or record count, and no records are excluded because of missing group keys.

In [ ]:
daily_range = price_new["high"] - price_new["low"]
print("Price records:", len(price_new))
print("Eligible daily ranges:", daily_range.count())
print("Missing daily ranges:", daily_range.isna().sum())
print("Missing symbols:", price_new["symbol"].isna().sum())
print("Missing dates:", price_new["date"].isna().sum())
range_preview = price_new[["id", "symbol", "date", "high", "low"]].copy()
range_preview["daily_range"] = daily_range
display(range_preview.nlargest(5, "daily_range"))


Price records: 2656
Eligible daily ranges: 2656
Missing daily ranges: 0
Missing symbols: 0
Missing dates: 0


,id,symbol,date,high,low,daily_range
1415,1416,META,2025-04-09,587.890015,502.109985,85.780029
1586,1587,META,2025-12-12,711.000000,638.609985,72.390015
2336,2337,TSLA,2024-12-18,488.540009,427.010010,61.529999
1413,1414,META,2025-04-07,539.369995,481.899994,57.470001
2450,2451,TSLA,2025-06-05,324.549988,273.209991,51.339996


## 12. Verify individual records

I selected five records from different datasets and companies to verify that the cleaning process worked as intended. Before checking the cleaned output, I expect each record’s date to represent the same date as the original record after conversion to datetime, while all other values remain unchanged.

Expected results:

1. The first AAPL filing record should retain the same ID, symbol, document type, and text, while its date is converted to datetime.
2. The first AAPL news record should retain the same ID, symbol, and highlights, while its date is converted to datetime.
3. The first AAPL price record should retain the same ID, symbol, and price/volume values, while its date is converted to datetime.
4. A META price record should retain its original values while its date is converted to datetime.
5. A TSLA price record should retain its original values while its date is converted to datetime.

In [ ]:
checks = [
    ("Filing", filings, filing_new, 1),
    ("News", news, news_new, 1),
    ("Price - AAPL", prices, price_new, 1),
    ("Price - META", prices, price_new, 1416),
    ("Price - TSLA", prices, price_new, 2337)
]

for name, original, cleaned, record_id in checks:
    print("\n", name, "- ID:", record_id)
    original_row = original[original["id"] == record_id]
    cleaned_row = cleaned[cleaned["id"] == record_id]

    
    display(original_row)
    display(cleaned_row)


 Filing - ID: 1


,id,symbol,date,document_type,mda_content,risk_content
0,1,AAPL,2024-02-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...


,id,symbol,date,document_type,mda_content,risk_content
0,1,AAPL,2024-02-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...



 News - ID: 1


,id,symbol,date,highlights
0,1,AAPL,2024-12-01,"Apple’s December 1, 2024 coverage centers on i..."


...

,id,symbol,date,open,high,low,close,adj_close,volume
1415,1416,META,2025-04-09,509.269989,587.890015,502.109985,585.77002,583.948242,39216600


,id,symbol,date,open,high,low,close,adj_close,volume
1415,1416,META,2025-04-09,509.269989,587.890015,502.109985,585.77002,583.948242,39216600



 Price - TSLA - ID: 2337


,id,symbol,date,open,high,low,close,adj_close,volume
2336,2337,TSLA,2024-12-18,466.5,488.540009,427.01001,440.130005,440.130005,149340800


,id,symbol,date,open,high,low,close,adj_close,volume
2336,2337,TSLA,2024-12-18,466.5,488.540009,427.01001,440.130005,440.130005,149340800


## 13. Test reproducibility


In [ ]:
import sys
import pyarrow as pa

print("Python:", sys.version)
print("pandas:", pd.__version__)
print("pyarrow:", pa.__version__)

Python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:23:37) [Clang 14.0.6 ]
pandas: 3.0.5
pyarrow: 24.0.0


## 14. Document remaining limitations

The cleaning checks identified no missing values, duplicate records, invalid dates, or invalid price/volume values in the three datasets. However, these checks focus on the structure and consistency of the data and do not verify the factual accuracy of the news highlights or filing content against their original sources.

In addition, daily price range is calculated as high − low in U.S. dollars per share. Because the eight companies have different stock price levels, the absolute daily ranges are not directly normalized across companies. This should be considered when comparing unusually wide price movements across companies.

## Cleaning decision log

This log summarizes the cleaning decisions and verification results.

| Rule | Reason | Check / evidence |
|---|---|---|
| Convert dates on working copies only | Dates were originally object-typed; analysis uses dates | Section 4: 0 failed date conversions and cleaned dates are datetime |
| Retain all rows | No missing values or duplicate rows/keys were found | Sections 6, 7, and 10: final row counts equal starting row counts |
| Retain price and volume values | No invalid or inconsistent values were found | Section 8: all suspicious-value checks returned 0 |
| Retain original text and IDs | Text and ID checks did not indicate changes were needed | Sections 5 and 12: original values were preserved |
| Preserve original inputs | Keep source data separate from cleaning | Section 3: cleaning was performed on copies of the original data |

## Codex or cleaning-decision verification

Codex suggested converting the date columns from object to datetime. I independently checked the original data types and confirmed that the date columns were stored as object. I converted the dates using pd.to_datetime() with errors="coerce" and checked for failed conversions. There were 0 failed conversions in the filings, news, and prices datasets, so I accepted the suggestion.

## Submission sample and repository information

Cleaned sample of up to 50 rows, including actual difficult cases: **[to select and prepare]**. Selection explanation / sharing restrictions and approved review channel, if applicable: **[to document]**.

Student name: **[to enter]**. Individual branch name: **[to enter]**. Branch URL: **[to enter]**. Submitted commit ID: **[to enter]**. Instructor repository access: **[to confirm]**. Final PDF and Gradescope submission: **[to complete]**.

Required external deliverables remain: cleaning `.py` script, notebook, data sample, dependency files, README, and individual final PDF. This edit does not create or update those files.


In [24]:
sample = price_new.copy()
sample["daily_range"] = sample["high"] - sample["low"]
sample = sample.nlargest(50, "daily_range")
sample.to_csv("cleaned_sample.csv", index=False)